# LeetCode 3402 — Minimum Operations to Make Columns Strictly Increasing bằng Python

Link đề: https://leetcode.com/problems/minimum-operations-to-make-columns-strictly-increasing/ · Độ khó: **Easy** · Chủ đề: Greedy (tham lam), Array, Matrix (ma trận)

Skeleton LeetCode:

```python
class Solution:
    def minimumOperations(self, grid: List[List[int]]) -> int:
        ...
```

Ghi chú: đề và ví dụ được dịch từ đề gốc (Weekly Contest 430); mọi lời giải trong notebook đã được chạy và kiểm chứng bằng code thật. Trong notebook ta viết `list[list[int]]` thay cho `List[List[int]]` để không cần `import`; khi nộp LeetCode cả hai đều chạy được.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: ma trận `m x n` gồm số **không âm**; mỗi thao tác **tăng một ô thêm 1**; cần số thao tác **ít nhất** để **mọi cột tăng chặt** từ trên xuống dưới (`grid[0][j] < grid[1][j] < ... < grid[m-1][j]`).
2. Nắm các công cụ Python: ma trận là **list của list** (`grid[i][j]`, `len(grid)`, `len(grid[0])`), **chuyển vị bằng `zip(*grid)`** để duyệt theo cột, sao chép sâu `[row[:] for row in grid]`, `max`, và `sum` trên biểu thức.
3. Đi từ cách **mô phỏng tăng từng đơn vị** → **tính ngay lượng cần tăng cho mỗi ô** → **duyệt theo cột bằng `zip(*grid)` với biến `pre`** → **không sửa input** → **đổi biến `a[i] - i` rồi lấy tối đa tiền tố**.
4. Hiểu mấu chốt tham lam: mỗi ô chỉ nên tăng **vừa đủ** để lớn hơn ô ngay phía trên (đã được chỉnh) đúng `1` đơn vị; tăng dư chỉ làm các ô bên dưới phải tăng thêm.
5. Tránh các lỗi kinh điển: duyệt theo **hàng** thay vì cột, so sánh với giá trị **cũ** (chưa cập nhật) của ô trên, tính lượng tăng thiếu `+1` (đòi hỏi `>` chứ không phải `>=`), quên `max(0, ...)` khi ô đã đủ lớn, và sửa trực tiếp input do sao chép nông.

## 2. Đề bài & ví dụ

Cho ma trận `grid` kích thước `m x n` gồm các số nguyên **không âm**. Trong **một thao tác** bạn có thể **tăng giá trị của bất kỳ ô `grid[i][j]` thêm `1`**.

Trả về **số thao tác tối thiểu** để **tất cả các cột** của `grid` đều **tăng chặt** (strictly increasing), tức là trong mỗi cột, đi từ trên xuống dưới thì mỗi số **lớn hơn hẳn** số ngay phía trên nó.

**Ví dụ 1**

- Input: `grid = [[3,2],[1,3],[3,4],[0,1]]`
- Output: `15`
- Giải thích: cột `0` là `[3,1,3,0]`: tăng `grid[1][0]` thêm `3` (thành `4`), `grid[2][0]` thêm `2` (thành `5`), `grid[3][0]` thêm `6` (thành `6`) ⇒ `11` thao tác. Cột `1` là `[2,3,4,1]`: tăng `grid[3][1]` thêm `4` (thành `5`) ⇒ `4` thao tác. Tổng `15`.

**Ví dụ 2**

- Input: `grid = [[3,2,1],[2,1,0],[1,2,3]]`
- Output: `12`
- Giải thích: cột `0` `[3,2,1]` cần `2 + 4 = 6`; cột `1` `[2,1,2]` cần `2 + 2 = 4`; cột `2` `[1,0,3]` cần `2` ⇒ tổng `12`.

**Ràng buộc:** `m == grid.length`, `n == grid[i].length`; `1 <= m, n <= 50`; `0 <= grid[i][j] < 2500`.

In [1]:
import random
from itertools import product

examples = [
    ([[3, 2], [1, 3], [3, 4], [0, 1]], 15),
    ([[3, 2, 1], [2, 1, 0], [1, 2, 3]], 12),
]
for grid, expected in examples:
    print(grid, "-> kỳ vọng", expected)

[[3, 2], [1, 3], [3, 4], [0, 1]] -> kỳ vọng 15
[[3, 2, 1], [2, 1, 0], [1, 2, 3]] -> kỳ vọng 12


## 3. Kiến thức nền

### 3.1. Ma trận là list của list: `grid[i][j]`

`grid` là một list; mỗi phần tử `grid[i]` là **một hàng** (cũng là list). `grid[i][j]` là ô ở **hàng `i`, cột `j`** (cả hai đếm từ `0`). Số hàng `m = len(grid)`, số cột `n = len(grid[0])`.

In [2]:
grid = [[3, 2],
        [1, 3],
        [3, 4],
        [0, 1]]
print(len(grid), len(grid[0]))     # 4 2  (4 hàng, 2 cột)
print(grid[1])                      # [1, 3]  (hàng 1)
print(grid[2][1])                   # 4       (hàng 2, cột 1)
col0 = [grid[i][0] for i in range(len(grid))]
print(col0)                         # [3, 1, 3, 0]  (cột 0, lấy bằng list comprehension)

4 2
[1, 3]
4
[3, 1, 3, 0]


### 3.2. Duyệt theo cột bằng `zip(*grid)` (chuyển vị)

Dấu `*` trước `grid` **trải** các hàng thành từng đối số riêng: `zip(*grid)` = `zip(hàng0, hàng1, hàng2, ...)`. `zip` ghép các phần tử **cùng vị trí** của các hàng, tức là cho ra **từng cột** dưới dạng tuple. Đây là phép **chuyển vị** (transpose) ma trận.

In [3]:
grid = [[3, 2], [1, 3], [3, 4], [0, 1]]
for j, col in enumerate(zip(*grid)):
    print(f"cột {j}:", col)
print(list(zip(*grid)))

cột 0: (3, 1, 3, 0)
cột 1: (2, 3, 4, 1)
[(3, 1, 3, 0), (2, 3, 4, 1)]


### 3.3. Sao chép ma trận: nông và sâu

`grid2 = grid[:]` chỉ sao chép **list ngoài**; các hàng bên trong vẫn là **cùng một đối tượng**. Sửa `grid2[0][0]` sẽ làm đổi cả `grid`. Muốn bản sao độc lập (sâu) dùng `[row[:] for row in grid]` (hoặc `copy.deepcopy`). Điều này quan trọng khi ta muốn mô phỏng mà **không phá input**.

In [4]:
grid = [[1, 2], [3, 4]]
shallow = grid[:]
deep = [row[:] for row in grid]
shallow[0][0] = 99
print(grid)      # [[99, 2], [3, 4]]  <- bị đổi theo bản sao nông!
deep[1][1] = -1
print(grid)      # vẫn [[99, 2], [3, 4]]: bản sao sâu không ảnh hưởng gốc
print(deep)      # [[99, 2], [3, -1]]

[[99, 2], [3, 4]]
[[99, 2], [3, 4]]
[[1, 2], [3, -1]]


### 3.4. `max`, `sum` và `+=` trên một cột

`max(a, b)` trả về số lớn hơn. `sum(biểu_thức for ...)` cộng dồn. `x += d` là `x = x + d`. Ta sẽ dùng `max(0, cần)` để không cộng số âm khi ô đã đủ lớn.

In [5]:
print(max(0, 3), max(0, -5))        # 3 0
print(sum(max(0, 5 - x) for x in [1, 7, 4]))   # (4) + 0 + (1) = 5
total = 0
for d in [2, 3, 4]:
    total += d
print(total)                          # 9

3 0
5
9


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Các cột **độc lập** với nhau: tăng một ô chỉ ảnh hưởng đến cột chứa nó. Vì vậy ta giải **từng cột** rồi cộng đáp án. Với một cột `[a0, a1, ..., a_{m-1}]` ta đi từ trên xuống:

- `a0` không có ô nào ở trên ⇒ giữ nguyên.
- `a1` phải **lớn hơn hẳn** giá trị (đã chỉnh) của ô trên. Nếu `a1 <= trên` thì tăng nó lên `trên + 1`, tốn `trên + 1 - a1` thao tác; ngược lại giữ nguyên.
- Cứ thế tiếp tục xuống dưới.

Cột `[3, 1, 3, 0]` của ví dụ 1:

| ô | giá trị | ô trên (sau chỉnh) | cần ít nhất | tăng thêm | giá trị mới |
|---|---|---|---|---|---|
| 0 | 3 | (không có) | — | 0 | 3 |
| 1 | 1 | 3 | 4 | 3 | 4 |
| 2 | 3 | 4 | 5 | 2 | 5 |
| 3 | 0 | 5 | 6 | 6 | 6 |

Tổng `0 + 3 + 2 + 6 = 11`. Cột `[2, 3, 4, 1]`: chỉ ô cuối cần tăng từ `1` lên `5` ⇒ `4`. Tổng `11 + 4 = 15` ✓.

In [6]:
col = [3, 1, 3, 0]
prev = None
total = 0
for x in col:
    need = 0 if prev is None else max(0, prev + 1 - x)
    new = x + need
    print(f"x={x}, trên={prev}, tăng {need} -> {new}")
    total += need
    prev = new
print("tổng cột:", total)

x=3, trên=None, tăng 0 -> 3
x=1, trên=3, tăng 3 -> 4
x=3, trên=4, tăng 2 -> 5
x=0, trên=5, tăng 6 -> 6
tổng cột: 11


## 5. Cách 1 — Brute Force / baseline: mô phỏng tăng từng đơn vị

**Ý tưởng.** Làm đúng như đề: với mỗi cột, đi từ hàng `1` xuống; **chừng nào** `grid[i][j] <= grid[i-1][j]` thì tăng `grid[i][j]` thêm `1` và đếm `1` thao tác. Ta làm việc trên **bản sao sâu** để không phá input.

In [7]:
def min_ops_simulate(grid: list[list[int]]) -> int:
    g = [row[:] for row in grid]                  # bản sao sâu
    m, n = len(g), len(g[0])
    ops = 0
    for j in range(n):                            # từng cột
        for i in range(1, m):                     # từng hàng, từ hàng 1 trở xuống
            while g[i][j] <= g[i - 1][j]:         # chưa lớn hơn hẳn ô trên
                g[i][j] += 1                      # tăng đúng 1 đơn vị
                ops += 1
    return ops


for grid, expected in examples:
    print(min_ops_simulate(grid), "(kỳ vọng", expected, ")")

15 (kỳ vọng 15 )
12 (kỳ vọng 12 )


**Độ phức tạp (Cách 1).** Gọi `V` là giá trị lớn nhất (`< 2500`). Ô cuối cột có thể phải tăng cỡ `V + m` lần ⇒ **Time `O(n · m · (V + m))`** trong trường hợp xấu (vẫn chạy được vì ràng buộc nhỏ), **Space `O(m·n)`** cho bản sao.

**Nút thắt:** ta tăng **từng đơn vị một** bằng vòng `while`, trong khi ta **biết sẵn** cần tăng bao nhiêu: `trên + 1 - x`. Có thể tính thẳng bằng một phép trừ.

## 6. Ý tưởng thuật toán / bước đột phá

**Tham lam theo từng ô.** Nếu ô trên (sau khi chỉnh) có giá trị `pre`, thì ô hiện tại `x` cần đạt **ít nhất** `pre + 1`:

- nếu `x > pre`: không cần làm gì, và **giá trị mới** là `x`;
- nếu `x <= pre`: tăng thêm `pre + 1 - x`, và giá trị mới là `pre + 1`.

Gọi `pre` là "giá trị chỉnh của ô trên". Khởi tạo `pre = -1` (nhỏ hơn mọi số không âm) thì ô đầu cột tự động thỏa `x > -1` mà không cần xử lý riêng. Biểu thức gọn:

```
need = max(0, pre + 1 - x)       # lượng cần tăng
pre = x + need = max(x, pre + 1) # giá trị mới
```

Vì sao **không tăng nhiều hơn mức vừa đủ** là tối ưu sẽ được chứng minh ở mục 10.

In [8]:
col = [2, 3, 4, 1]
pre = -1
total = 0
for x in col:
    need = max(0, pre + 1 - x)
    pre = x + need
    total += need
    print(f"x={x}: need={need}, giá trị mới={pre}, tổng={total}")

x=2: need=0, giá trị mới=2, tổng=0
x=3: need=0, giá trị mới=3, tổng=0
x=4: need=0, giá trị mới=4, tổng=0
x=1: need=4, giá trị mới=5, tổng=4


## 7. Các cách tiếp theo

### Cách 2 — Tính trực tiếp lượng cần tăng, duyệt theo chỉ số

**Giải thích từng đoạn.**

1. Sao chép sâu `g = [row[:] for row in grid]` để có thể ghi giá trị mới vào ô.
2. Với mỗi cột `j` và hàng `i` từ `1`: `need = g[i-1][j] + 1 - g[i][j]`.
3. Nếu `need > 0` thì `ops += need` và `g[i][j] += need` (**cập nhật** để ô dưới so với giá trị mới).

In [9]:
def min_ops_index(grid: list[list[int]]) -> int:
    g = [row[:] for row in grid]
    m, n = len(g), len(g[0])
    ops = 0
    for j in range(n):
        for i in range(1, m):
            need = g[i - 1][j] + 1 - g[i][j]       # cần tăng để > ô trên
            if need > 0:
                ops += need
                g[i][j] += need                    # cập nhật để ô dưới so với giá trị mới
    return ops


for grid, expected in examples:
    print(min_ops_index(grid), "(kỳ vọng", expected, ")")

15 (kỳ vọng 15 )
12 (kỳ vọng 12 )


In [10]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionIndex:
    def minimumOperations(self, grid: list[list[int]]) -> int:
        g = [row[:] for row in grid]
        m, n = len(g), len(g[0])
        ops = 0
        for j in range(n):
            for i in range(1, m):
                need = g[i - 1][j] + 1 - g[i][j]
                if need > 0:
                    ops += need
                    g[i][j] += need
        return ops


print(SolutionIndex().minimumOperations([[3, 2], [1, 3], [3, 4], [0, 1]]))      # 15

15


**Độ phức tạp Cách 2:** **Time `O(m·n)`**, **Space `O(m·n)`** (bản sao `g`).

### Cách 3 — Duyệt cột bằng `zip(*grid)` và biến `pre` (cách gọn, nên nộp)

**Giải thích.** `zip(*grid)` cho từng cột; trong mỗi cột giữ một số `pre` là giá trị chỉnh của ô trên (khởi tạo `-1`). Với mỗi `x`: nếu `pre < x` thì chỉ cập nhật `pre = x`; ngược lại `ops += pre + 1 - x` và `pre += 1`. **Không cần sao chép ma trận**.

In [11]:
def min_ops_zip(grid: list[list[int]]) -> int:
    ops = 0
    for col in zip(*grid):                  # col là một cột (tuple)
        pre = -1                            # giá trị chỉnh của ô trên; -1 < mọi số >= 0
        for x in col:
            if pre < x:                     # đã lớn hơn hẳn ô trên
                pre = x
            else:                           # x <= pre: phải nâng lên pre + 1
                pre += 1
                ops += pre - x
    return ops


for grid, expected in examples:
    print(min_ops_zip(grid), "(kỳ vọng", expected, ")")

15 (kỳ vọng 15 )
12 (kỳ vọng 12 )


In [12]:
# Code nộp LeetCode cho Cách 3
class SolutionZip:
    def minimumOperations(self, grid: list[list[int]]) -> int:
        ops = 0
        for col in zip(*grid):
            pre = -1
            for x in col:
                if pre < x:
                    pre = x
                else:
                    pre += 1
                    ops += pre - x
        return ops


print(SolutionZip().minimumOperations([[3, 2, 1], [2, 1, 0], [1, 2, 3]]))        # 12

12


**Độ phức tạp Cách 3:** **Time `O(m·n)`** (mỗi ô một lần), **Space `O(m·n)` cho `zip(*grid)`** nếu tính cả các tuple cột tạo ra (thực tế `zip` sinh từng cột lười biếng nên mỗi lúc chỉ giữ một cột: `O(m)`).

### Cách 4 — Duyệt theo chỉ số, không sao chép, không `zip`

**Giải thích.** Nếu không muốn dùng `zip(*grid)`, ta vẫn không cần sao chép ma trận: giữ `pre` cho từng cột bằng vòng lặp ngoài là cột `j`, vòng trong là hàng `i`, đọc `grid[i][j]` và **chỉ cập nhật biến `pre`**, không ghi lại vào ma trận. Cách này giữ nguyên input và dùng bộ nhớ phụ `O(1)`.

In [13]:
def min_ops_no_copy(grid: list[list[int]]) -> int:
    m, n = len(grid), len(grid[0])
    ops = 0
    for j in range(n):
        pre = -1
        for i in range(m):
            x = grid[i][j]
            new = max(x, pre + 1)           # giá trị nhỏ nhất thỏa > pre mà >= x
            ops += new - x
            pre = new
    return ops


for grid, expected in examples:
    print(min_ops_no_copy(grid), "(kỳ vọng", expected, ")")

15 (kỳ vọng 15 )
12 (kỳ vọng 12 )


In [14]:
# Code nộp LeetCode cho Cách 4
class SolutionNoCopy:
    def minimumOperations(self, grid: list[list[int]]) -> int:
        m, n = len(grid), len(grid[0])
        ops = 0
        for j in range(n):
            pre = -1
            for i in range(m):
                new = max(grid[i][j], pre + 1)
                ops += new - grid[i][j]
                pre = new
        return ops


print(SolutionNoCopy().minimumOperations([[3, 2], [1, 3], [3, 4], [0, 1]]))     # 15

15


**Độ phức tạp Cách 4:** **Time `O(m·n)`**, **Space `O(1)`**.

### Cách 5 — Đổi biến `b[i] = a[i] - i`: từ "tăng chặt" thành "không giảm"

**Giải thích.** Dãy `a` **tăng chặt** (số nguyên) ⇔ `a[i] >= a[i-1] + 1` ⇔ `a[i] - i >= a[i-1] - (i-1)`. Đặt `b[i] = a[i] - i`; khi đó `a` tăng chặt ⇔ `b` **không giảm**. Để làm `b` không giảm bằng cách chỉ **tăng**, mỗi `b[i]` phải nâng lên bằng **giá trị lớn nhất trong `b[0..i]`** (tối đa tiền tố). Chi phí của ô `i` là `max_prefix(b)[i] - b[i]`. Đây là phiên bản "tối đa tiền tố" của cùng một thuật toán tham lam (mẫu cho bài tương tự "làm dãy không giảm").

In [15]:
def min_ops_prefix_max(grid: list[list[int]]) -> int:
    ops = 0
    for col in zip(*grid):
        best = None                          # tối đa tiền tố của b[i] = col[i] - i
        for i, x in enumerate(col):
            b = x - i
            best = b if best is None else max(best, b)
            ops += best - b                  # nâng b lên best
    return ops


for grid, expected in examples:
    print(min_ops_prefix_max(grid), "(kỳ vọng", expected, ")")

15 (kỳ vọng 15 )
12 (kỳ vọng 12 )


**Độ phức tạp Cách 5:** **Time `O(m·n)`**, **Space `O(1)`** phụ (ngoài `zip`).

## 8. TRACE từng bước

Ta trace Cách 3 trên `grid = [[3,2],[1,3],[3,4],[0,1]]`. Bảng cho **cột 0** `(3,1,3,0)`:

| bước | `x` | `pre` trước | `pre < x`? | hành động | `ops` sau |
|---|---|---|---|---|---|
| 1 | 3 | -1 | có | `pre = 3` | 0 |
| 2 | 1 | 3 | không | `pre = 4`, `ops += 4 - 1 = 3` | 3 |
| 3 | 3 | 4 | không | `pre = 5`, `ops += 5 - 3 = 2` | 5 |
| 4 | 0 | 5 | không | `pre = 6`, `ops += 6 - 0 = 6` | 11 |

**Cột 1** `(2,3,4,1)`: `2`, `3`, `4` đều lớn hơn ô trên nên chỉ cập nhật `pre`; đến `x = 1` có `pre = 4` nên `pre = 5`, `ops += 5 - 1 = 4` ⇒ `15`.

In [16]:
def trace_min_ops(grid):
    ops = 0
    for j, col in enumerate(zip(*grid)):
        pre = -1
        print(f"--- cột {j}: {col}")
        for x in col:
            if pre < x:
                print(f"x={x}, pre={pre}: pre < x -> giữ nguyên, pre = {x}")
                pre = x
            else:
                pre += 1
                ops += pre - x
                print(f"x={x}: x <= pre -> nâng lên {pre}, tốn {pre - x}, ops = {ops}")
    print("kết quả:", ops)
    return ops


trace_min_ops([[3, 2], [1, 3], [3, 4], [0, 1]])

--- cột 0: (3, 1, 3, 0)
x=3, pre=-1: pre < x -> giữ nguyên, pre = 3
x=1: x <= pre -> nâng lên 4, tốn 3, ops = 3
x=3: x <= pre -> nâng lên 5, tốn 2, ops = 5
x=0: x <= pre -> nâng lên 6, tốn 6, ops = 11
--- cột 1: (2, 3, 4, 1)
x=2, pre=-1: pre < x -> giữ nguyên, pre = 2
x=3, pre=2: pre < x -> giữ nguyên, pre = 3
x=4, pre=3: pre < x -> giữ nguyên, pre = 4
x=1: x <= pre -> nâng lên 5, tốn 4, ops = 15
kết quả: 15


15

## 9. Code nộp LeetCode

Ta chọn **Cách 3** (`zip(*grid)` + `pre`): ngắn, không sửa input, `O(m·n)`.

In [17]:
class Solution:
    def minimumOperations(self, grid: list[list[int]]) -> int:
        ops = 0
        for col in zip(*grid):          # từng cột (chuyển vị)
            pre = -1                    # giá trị (đã chỉnh) của ô phía trên; -1 < mọi số >= 0
            for x in col:
                if pre < x:             # đã lớn hơn hẳn ô trên: giữ nguyên
                    pre = x
                else:                   # phải nâng lên pre + 1 (vừa đủ)
                    pre += 1
                    ops += pre - x
        return ops


for grid, expected in examples:
    print(Solution().minimumOperations(grid), "(kỳ vọng", expected, ")")

15 (kỳ vọng 15 )
12 (kỳ vọng 12 )


## 10. Vì sao đúng

- **Các cột độc lập.** Mỗi thao tác chỉ tăng một ô nên chỉ ảnh hưởng một cột; điều kiện "tăng chặt" cũng chỉ so các ô **cùng cột**. Tổng chi phí tối thiểu `=` tổng chi phí tối thiểu của từng cột.
- **Một cột: tham lam.** Gọi `t_i` là giá trị cuối cùng của ô `i` (với `t_i >= a_i`, vì chỉ được tăng) và cần `t_i >= t_{i-1} + 1`. Ta chứng minh **mọi lời giải hợp lệ có `t_i >= p_i`**, trong đó `p_i = max(a_i, p_{i-1} + 1)` là giá trị của thuật toán tham lam. Quy nạp: `p_0 = a_0 <= t_0`. Giả sử `t_{i-1} >= p_{i-1}`; thì `t_i >= t_{i-1} + 1 >= p_{i-1} + 1` và `t_i >= a_i`, nên `t_i >= max(a_i, p_{i-1} + 1) = p_i`. Chi phí `Σ (t_i - a_i) >= Σ (p_i - a_i)`.
- **Tham lam đạt được cận dưới.** Dãy `p_i` thật sự hợp lệ (`p_i >= a_i`, `p_i >= p_{i-1} + 1`), nên chi phí `Σ (p_i - a_i)` là **đạt được** ⇒ tối ưu.
- **Khởi tạo `pre = -1`.** Mọi `a_0 >= 0 > -1` nên ô đầu cột giữ nguyên (`p_0 = a_0`), không cần xử lý đặc biệt.
- **Khớp Cách 5.** `p_i - i = max_{k <= i}(a_k - k)` — đúng công thức tối đa tiền tố.

## 11. Time/Space Complexity

Gọi `m` là số hàng, `n` là số cột.

- **Time `O(m·n)`**: mỗi ô được xét đúng một lần với phép tính `O(1)`. Tối đa `50 × 50 = 2500` ô.
- **Space `O(m)`** (hoặc `O(1)` nếu duyệt bằng chỉ số): chỉ giữ cột hiện tại và biến `pre`, `ops`.

Lưu ý: tổng số thao tác có thể lớn (cột `m = 50` ô cùng `0` cần `0+1+...+49 = 1225` thao tác; nhiều cột hơn nữa), nhưng Python có số nguyên không giới hạn nên không lo tràn số.

## 12. Bộ test

### 12.1. Ví dụ của đề và gom các cách vào một bảng

In [18]:
impls = {
    "simulate": min_ops_simulate,
    "index": min_ops_index,
    "zip": min_ops_zip,
    "no_copy": min_ops_no_copy,
    "prefix_max": min_ops_prefix_max,
    "Solution": lambda grid: Solution().minimumOperations(grid),
}

for grid, expected in examples:
    for name, fn in impls.items():
        assert fn(grid) == expected, (name, grid)
print("PASS các ví dụ của đề với", len(impls), "cách")

PASS các ví dụ của đề với 6 cách


### 12.2. Edge case

- Một hàng (`m = 1`): luôn `0` thao tác vì không có cặp ô liền nhau.
- Một cột (`n = 1`) và ma trận `1 x 1`.
- Cột **đã tăng chặt** sẵn (đáp án `0`), cột **toàn số `0`** (cần `0+1+...+(m-1)`), cột **giảm dần**.
- Ô **bằng** ô trên (đòi hỏi `+1` vì phải `>`, không phải `>=`).
- Giá trị lớn gần `2499`.

In [19]:
cases = [
    ([[5]], 0),
    ([[1, 2, 3]], 0),                        # một hàng
    ([[0], [0], [0]], 0 + 1 + 2),            # một cột toàn 0: cần 0,1,2
    ([[1], [1]], 1),                         # bằng nhau cũng phải tăng
    ([[1], [2], [3]], 0),                    # đã tăng chặt
    ([[3], [2], [1]], 2 + 4),                # giảm dần: 3,4,5 -> tốn 0 + 2 + 4
    ([[0, 5], [0, 5], [0, 5]], 3 + 3),       # hai cột, mỗi cột cần 0+1+2 = 3
    ([[2499], [0]], 2500),                   # nâng 0 lên 2500
    ([[0], [2499]], 0),
    ([[1, 9], [2, 3], [3, 1]], 17),          # cột 0 đã tăng; cột 1 [9,3,1] -> 9,10,11 tốn 7 + 10
]
for grid, expected in cases:
    for name, fn in impls.items():
        assert fn(grid) == expected, (name, grid, fn(grid), expected)
print("PASS", len(cases), "edge case x", len(impls), "cách")

PASS 10 edge case x 6 cách


### 12.3. Kiểm tra tính hợp lệ và tối ưu bằng vét cạn

Hai kiểm tra **độc lập** với thuật toán tham lam:

1. **Dựng ma trận kết quả** từ tham lam rồi kiểm tra: mọi cột tăng chặt, mọi ô `>=` giá trị gốc, tổng chênh lệch `= ops`.
2. **Vét cạn** trên cột nhỏ: thử mọi cách chọn giá trị cuối `t_i` trong một miền nhỏ và lấy tổng chi phí nhỏ nhất hợp lệ.

In [20]:
def build_final(grid):
    """Dựng ma trận sau khi tăng theo tham lam."""
    m, n = len(grid), len(grid[0])
    final = [[0] * n for _ in range(m)]
    for j in range(n):
        pre = -1
        for i in range(m):
            final[i][j] = max(grid[i][j], pre + 1)
            pre = final[i][j]
    return final


def is_valid_result(grid, final, ops):
    m, n = len(grid), len(grid[0])
    for j in range(n):
        for i in range(m):
            if final[i][j] < grid[i][j]:
                return False                                  # chỉ được tăng
            if i > 0 and not final[i - 1][j] < final[i][j]:
                return False                                  # cột phải tăng chặt
    return sum(final[i][j] - grid[i][j] for i in range(m) for j in range(n)) == ops


def brute_force_column(col, limit):
    """Chi phí nhỏ nhất thử mọi dãy t_i trong [0, limit]."""
    best = None
    for t in product(range(limit + 1), repeat=len(col)):
        if any(t[i] < col[i] for i in range(len(col))):
            continue
        if any(t[i] <= t[i - 1] for i in range(1, len(col))):
            continue
        cost = sum(t[i] - col[i] for i in range(len(col)))
        if best is None or cost < best:
            best = cost
    return best


grid = [[3, 2], [1, 3], [3, 4], [0, 1]]
final = build_final(grid)
print(final)
assert is_valid_result(grid, final, Solution().minimumOperations(grid))

for length in range(1, 5):
    for col in product(range(4), repeat=length):
        expected = brute_force_column(col, limit=3 + length)         # miền đủ rộng: max + m
        got = Solution().minimumOperations([[x] for x in col])
        assert got == expected, (col, got, expected)
print("PASS vét cạn mọi cột độ dài 1..4 với giá trị 0..3")

[[3, 2], [4, 3], [5, 4], [6, 5]]


PASS vét cạn mọi cột độ dài 1..4 với giá trị 0..3


### 12.4. Đối chiếu ngẫu nhiên (có seed) đúng ràng buộc

Với kích thước **nhỏ** ta đối chiếu cả cách mô phỏng (chậm). Với kích thước **lớn** (`m, n` tới `50`, giá trị `< 2500`) ta đối chiếu các cách nhanh với nhau và kiểm tra tính hợp lệ.

In [21]:
rng = random.Random(3402)
for trial in range(2000):                               # kích thước nhỏ: có cả mô phỏng
    m, n = rng.randint(1, 6), rng.randint(1, 6)
    grid = [[rng.randint(0, 20) for _ in range(n)] for _ in range(m)]
    ref = min_ops_simulate(grid)
    for name, fn in impls.items():
        assert fn(grid) == ref, (name, grid)
print("PASS 2000 ma trận nhỏ (đối chiếu cả mô phỏng)")

fast = {k: v for k, v in impls.items() if k != "simulate"}
for trial in range(300):                                # kích thước lớn đúng ràng buộc
    m, n = rng.randint(1, 50), rng.randint(1, 50)
    hi = rng.choice([1, 10, 100, 2499])
    grid = [[rng.randint(0, hi) for _ in range(n)] for _ in range(m)]
    assert len(grid) == m and all(len(r) == n for r in grid) and all(0 <= x < 2500 for r in grid for x in r)
    before = [row[:] for row in grid]
    ref = min_ops_prefix_max(grid)
    for name, fn in fast.items():
        assert fn(grid) == ref, (name,)
    assert grid == before, "input không được bị thay đổi"
    assert is_valid_result(grid, build_final(grid), ref)
print("PASS 300 ma trận lớn đúng ràng buộc; input không bị sửa")

PASS 2000 ma trận nhỏ (đối chiếu cả mô phỏng)


PASS 300 ma trận lớn đúng ràng buộc; input không bị sửa


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Duyệt theo hàng thay vì theo cột

In [22]:
def sai_theo_hang(grid):
    ops = 0
    for row in grid:                    # SAI: mỗi hàng là một "cột" của bài khác
        pre = -1
        for x in row:
            if pre < x:
                pre = x
            else:
                pre += 1
                ops += pre - x
    return ops


grid = [[3, 2], [1, 3], [3, 4], [0, 1]]
print("sai:", sai_theo_hang(grid), "| đúng:", Solution().minimumOperations(grid))

sai: 2 | đúng: 15


**Vì sao sai:** đề yêu cầu **mỗi cột** tăng chặt từ trên xuống, tức so `grid[i][j]` với `grid[i-1][j]` (ô **ngay trên**). Duyệt từng `row` lại so sánh các ô **cùng hàng** (trái-phải). **Cách sửa:** dùng `zip(*grid)` để lấy cột, hoặc vòng ngoài là cột `j`, vòng trong là hàng `i`.

### Lỗi 2: So sánh với giá trị CŨ của ô trên (không cập nhật sau khi tăng)

In [23]:
def sai_khong_cap_nhat(grid):
    m, n = len(grid), len(grid[0])
    ops = 0
    for j in range(n):
        for i in range(1, m):
            if grid[i][j] <= grid[i - 1][j]:
                ops += grid[i - 1][j] + 1 - grid[i][j]      # SAI: ô trên đã được tăng nhưng ta dùng giá trị gốc
    return ops


print("sai:", sai_khong_cap_nhat([[0], [0], [0]]), "| đúng:", Solution().minimumOperations([[0], [0], [0]]))
print("sai:", sai_khong_cap_nhat([[3], [1], [3], [0]]), "| đúng:", Solution().minimumOperations([[3], [1], [3], [0]]))

sai: 2 | đúng: 3
sai: 7 | đúng: 11


**Vì sao sai:** sau khi tăng ô `i`, giá trị mới của nó (không phải giá trị gốc) mới là mốc cho ô `i + 1`. Với `[0,0,0]` ta cần `0,1,2` (tốn `3`) nhưng cách sai chỉ tính `1 + 1 = 2`. **Cách sửa:** cập nhật `g[i][j] += need`, hoặc giữ `pre` là giá trị **đã chỉnh**.

### Lỗi 3: Thiếu `+1` (đòi `>=` thay vì `>`)

In [24]:
def sai_thieu_cong_1(grid):
    ops = 0
    for col in zip(*grid):
        pre = -1
        for x in col:
            if pre < x:
                pre = x
            else:
                ops += pre - x                 # SAI: thiếu +1, mới chỉ làm x == pre
                # (pre giữ nguyên)
    return ops


print("sai:", sai_thieu_cong_1([[1], [1]]), "| đúng:", Solution().minimumOperations([[1], [1]]))
print("sai:", sai_thieu_cong_1([[3, 2], [1, 3], [3, 4], [0, 1]]), "| đúng:", Solution().minimumOperations([[3, 2], [1, 3], [3, 4], [0, 1]]))

sai: 0 | đúng: 1
sai: 8 | đúng: 15


**Vì sao sai:** đề yêu cầu **tăng chặt** (`<`), nên hai ô bằng nhau vẫn **chưa** hợp lệ: cần `pre + 1 - x` thao tác, không phải `pre - x`. **Cách sửa:** `pre += 1` trước, rồi `ops += pre - x`.

### Lỗi 4: Quên `max(0, ...)` khi ô đã đủ lớn (ra số âm)

In [25]:
def sai_so_am(grid):
    ops = 0
    for col in zip(*grid):
        pre = -1
        for x in col:
            need = pre + 1 - x                  # SAI: có thể âm khi x đã lớn hơn pre + 1
            ops += need
            pre = x + need
    return ops


print("sai:", sai_so_am([[1], [5]]), "| đúng:", Solution().minimumOperations([[1], [5]]))

sai: -5 | đúng: 0


**Vì sao sai:** với cột `[1, 5]`, `need = 1 + 1 - 5 = -3`: ta "giảm" ô `5` — thao tác này **không được phép** (chỉ được tăng). **Cách sửa:** `need = max(0, pre + 1 - x)` hoặc nhánh `if pre < x` để giữ nguyên.

### Lỗi 5: Sao chép nông làm hỏng input

In [26]:
def sai_sao_chep_nong(grid):
    g = grid[:]                                   # SAI: chỉ sao chép list ngoài, các hàng vẫn dùng chung
    m, n = len(g), len(g[0])
    ops = 0
    for j in range(n):
        for i in range(1, m):
            need = g[i - 1][j] + 1 - g[i][j]
            if need > 0:
                ops += need
                g[i][j] += need
    return ops


grid = [[3, 2], [1, 3], [3, 4], [0, 1]]
print(sai_sao_chep_nong(grid), "->", "input đã bị đổi:", grid)

15 -> input đã bị đổi: [[3, 2], [4, 3], [5, 4], [6, 5]]


**Vì sao sai:** `grid[:]` không sao chép các hàng bên trong, nên `g[i][j] += need` ghi **thẳng vào input**. Trên LeetCode nhiều bài không cho phép/không mong muốn sửa input. **Cách sửa:** `[row[:] for row in grid]`, hoặc tốt hơn là không ghi gì (chỉ dùng biến `pre`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Mỗi cột/hàng phải tăng (hoặc không giảm) và chỉ được tăng giá trị**" ⇒ **tham lam** từng ô, dùng biến `pre` là giá trị đã chỉnh của ô trước.
- Các hàng/cột **độc lập** ⇒ giải từng cột rồi cộng.
- Tăng **chặt** (`<`) với số nguyên ⇔ `a[i] - i` **không giảm**; không giảm ⇔ lấy tối đa tiền tố.
- Khởi tạo `pre = -1` khi mọi giá trị `>= 0` để khỏi xử lý riêng phần tử đầu.
- `zip(*grid)` là cách ngắn nhất để duyệt **theo cột**; nhớ nó trả về **tuple**.
- Sao chép ma trận: `[row[:] for row in grid]`, **không** `grid[:]`.
- Luôn tự đối chiếu với **mô phỏng chậm** (tăng từng đơn vị) trên ma trận nhỏ để chắc công thức đúng.

## 15. Biến thể của bài

1. **Tăng chặt theo hàng** (trái sang phải) — chỉ cần đổi `zip(*grid)` thành `grid`.
2. **Không giảm** (`<=`) thay vì tăng chặt: bỏ `+1`, `pre = max(pre, x)` và `ops += pre - x`.
3. **Được tăng hoặc giảm** một đơn vị mỗi thao tác, làm cột **không giảm** ở chi phí nhỏ nhất ⇒ trung vị/slope trick (khó hơn).
4. **Trả về ma trận kết quả** (không chỉ số thao tác): chính là `build_final` ở mục 12.3.
5. Bài liên quan: **Minimum Operations to Make the Array Increasing (1827)**, **Make Array Strictly Increasing (1187)**, **Minimum Increment to Make Array Unique (945)**, **Non-decreasing Array (665)**.

In [27]:
def min_ops_rows(grid):
    ops = 0
    for row in grid:                                  # tăng chặt theo hàng
        pre = -1
        for x in row:
            new = max(x, pre + 1)
            ops += new - x
            pre = new
    return ops


def min_ops_non_decreasing_cols(grid):
    ops = 0
    for col in zip(*grid):                            # không giảm theo cột
        pre = 0
        for x in col:
            new = max(x, pre)
            ops += new - x
            pre = new
    return ops


def min_ops_array_increasing(nums):                   # LeetCode 1827: một cột
    return Solution().minimumOperations([[x] for x in nums])


print(min_ops_rows([[3, 2], [1, 3]]))                          # hàng [3,2] cần 2->4 (2), hàng [1,3] cần 0 => 2
print(min_ops_non_decreasing_cols([[3, 2], [1, 3], [3, 4], [0, 1]]))   # cột0: 3,3,3,3 tốn 5; cột1: 2,3,4,4 tốn 3 -> 8
print(min_ops_array_increasing([1, 1, 1]))                     # 3
assert min_ops_rows([[3, 2], [1, 3]]) == 2
assert min_ops_non_decreasing_cols([[3, 2], [1, 3], [3, 4], [0, 1]]) == 5 + 3
assert min_ops_array_increasing([1, 1, 1]) == 3

2
8
3


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | tăng từng đơn vị bằng `while` | `O(m·n·(V+m))` | `O(m·n)` | đáp án chuẩn cho ma trận nhỏ |
| 2. Tính `need` theo chỉ số | ghi giá trị mới vào bản sao | `O(m·n)` | `O(m·n)` | dễ hiểu, nhưng tốn bản sao |
| 3. `zip(*grid)` + `pre` | duyệt cột, giữ giá trị chỉnh | `O(m·n)` | `O(m)` | **nên nộp** |
| 4. Chỉ số, không sao chép | `new = max(x, pre + 1)` | `O(m·n)` | `O(1)` | khi cần không tạo tuple cột |
| 5. Đổi biến `a[i] - i` | tối đa tiền tố | `O(m·n)` | `O(1)` | hiểu bản chất "không giảm" |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `list(zip(*[[1, 2], [3, 4], [5, 6]]))`, `max(0, 2 + 1 - 9)`, `len([[1, 2, 3], [4, 5, 6]][0])`, `Solution().minimumOperations([[1], [1], [1]])`, `Solution().minimumOperations([[5, 0]])`.

**Bài 2 (tính tay).** Với `grid = [[2, 7], [2, 1], [9, 1]]`: xử lý từng cột, ghi giá trị mới của từng ô và tổng thao tác.

**Bài 3 (điền TODO).** Hoàn thành `min_ops_todo(grid)` theo Cách 3.

In [28]:
def min_ops_todo(grid):
    ops = 0
    # TODO 1: duyệt từng cột col trong zip(*grid)
    #   TODO 2: pre = -1
    #   TODO 3: duyệt x trong col: nếu pre < x thì pre = x
    #           ngược lại pre += 1 và ops += pre - x
    # TODO 4: trả về ops
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `min_ops_final_grid(grid)` trả về **cả ma trận kết quả** sau các thao tác tối thiểu (không sửa `grid` gốc) và số thao tác, dùng bản sao sâu.

**Bài 5 (biến thể / mini challenge).** (a) Viết `min_ops_rows_and_cols(grid)` đếm thao tác để **cả hàng lẫn cột** đều tăng chặt — gợi ý: duyệt từng ô theo thứ tự hàng rồi cột và đặt `new = max(x, trên + 1, trái + 1)`; chứng minh lý do tham lam vẫn tối ưu. (b) Viết `is_strictly_increasing_columns(grid)` kiểm tra một ma trận đã thỏa điều kiện của đề chưa.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[(1, 3, 5), (2, 4, 6)]`, `0`, `3`, `3` (cột `1,1,1` ⇒ `1,2,3`: `0 + 1 + 2`), `0` (một hàng, không có cặp ô liền nhau theo cột). Bài 3: bản sao của Cách 3 ở mục 9.
- **Mức 2 (vừa):** Bài 2: cột `0` là `(2,2,9)`: `2` giữ; `2 -> 3` (tốn `1`); `9` giữ ⇒ `1`. Cột `1` là `(7,1,1)`: `7` giữ; `1 -> 8` (tốn `7`); `1 -> 9` (tốn `8`) ⇒ `15`. Tổng `16`. Bài 4: `final[i][j] = max(grid[i][j], pre + 1)` rồi `pre = final[i][j]`.
- **Mức 3 (sâu):** Bài 5(a): `new = max(x, trên + 1, trái + 1)` với `trên`, `trái` là giá trị **đã chỉnh** (hàng trên, ô bên trái); với `-1` cho biên. Lý do: ô nhỏ nhất thỏa cả hai ràng buộc **và** `>= x` là `max` của ba chặn dưới, và giá trị càng nhỏ càng không làm các ô sau tốn thêm. 5(b): kiểm tra `grid[i][j] > grid[i-1][j]` với mọi `i >= 1`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [29]:
# Bài 1
assert list(zip(*[[1, 2], [3, 4], [5, 6]])) == [(1, 3, 5), (2, 4, 6)]
assert max(0, 2 + 1 - 9) == 0
assert len([[1, 2, 3], [4, 5, 6]][0]) == 3
assert Solution().minimumOperations([[1], [1], [1]]) == 3
assert Solution().minimumOperations([[5, 0]]) == 0

# Bài 2
assert Solution().minimumOperations([[2, 7], [2, 1], [9, 1]]) == 1 + 15


# Bài 3
def min_ops_todo_ans(grid):
    ops = 0
    for col in zip(*grid):
        pre = -1
        for x in col:
            if pre < x:
                pre = x
            else:
                pre += 1
                ops += pre - x
    return ops


for grid, expected in examples:
    assert min_ops_todo_ans(grid) == expected


# Bài 4
def min_ops_final_grid(grid):
    final = [row[:] for row in grid]
    m, n = len(final), len(final[0])
    ops = 0
    for j in range(n):
        pre = -1
        for i in range(m):
            new = max(final[i][j], pre + 1)
            ops += new - final[i][j]
            final[i][j] = new
            pre = new
    return final, ops


g0 = [[3, 2], [1, 3], [3, 4], [0, 1]]
final, ops = min_ops_final_grid(g0)
assert ops == 15 and g0 == [[3, 2], [1, 3], [3, 4], [0, 1]]       # input không bị sửa
assert final == [[3, 2], [4, 3], [5, 4], [6, 5]]


# Bài 5
def min_ops_rows_and_cols(grid):
    m, n = len(grid), len(grid[0])
    final = [row[:] for row in grid]
    ops = 0
    for i in range(m):
        for j in range(n):
            up = final[i - 1][j] if i > 0 else -1
            left = final[i][j - 1] if j > 0 else -1
            new = max(final[i][j], up + 1, left + 1)
            ops += new - final[i][j]
            final[i][j] = new
    return ops


def is_strictly_increasing_columns(grid):
    return all(grid[i][j] > grid[i - 1][j] for i in range(1, len(grid)) for j in range(len(grid[0])))


assert min_ops_rows_and_cols([[1, 1], [1, 1]]) == 0 + 1 + 1 + 2     # 1 2 / 2 3
assert is_strictly_increasing_columns([[3, 2], [4, 3], [5, 4], [6, 5]])
assert not is_strictly_increasing_columns([[3, 2], [1, 3]])
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (tham lam từng cột: nâng mỗi ô lên vừa đủ lớn hơn ô trên)

```python
class Solution:
    def minimumOperations(self, grid: List[List[int]]) -> int:
        ops = 0
        for col in zip(*grid):
            pre = -1
            for x in col:
                if pre < x:
                    pre = x
                else:
                    pre += 1
                    ops += pre - x
        return ops
```

**Checklist**

- [ ] Duyệt theo **cột** (`zip(*grid)`), so với ô **ngay trên**?
- [ ] `pre` là giá trị **đã chỉnh** của ô trên (cập nhật sau mỗi ô)?
- [ ] Đòi hỏi `>` nên tăng lên `pre + 1` (không phải `pre`)?
- [ ] Không bao giờ cộng số âm (nhánh `pre < x` giữ nguyên)?
- [ ] Không sửa input (dùng bản sao sâu hoặc chỉ dùng biến `pre`)?
- [ ] Đã thử `m = 1`, cột toàn `0`, `[[1],[1]]` (đáp án `1`), và hai ví dụ của đề (`15`, `12`)?
- [ ] Skeleton đúng `class Solution` / `def minimumOperations(self, grid: List[List[int]]) -> int`?